In [1]:
import os
from typing import cast
import transformers

import numpy as np
import pandas as pd
import torch
import transformers
from datasets import Dataset as HfDataset, DatasetDict, Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
AutoModelForSequenceClassification,
AutoTokenizer,
DataCollatorWithPadding,
Trainer,
TrainingArguments,
)

print(transformers.__version__)


5.3.0


# =========================
# 1. Load
# =========================

In [2]:
# =========================
# 1. Carregar dataset
# =========================
path = "/Users/jose-cleiton/dev/script_pncp/data/treinamento/2026-03-14/dataset.csv"
df = pd.read_csv(path)

df = df.rename(columns={"Objeto": "text", "Relevante": "label"})

df["label"] = (
    df["label"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", "", regex=True)
)

df = df[df["label"].isin(["0", "1"])].copy()
df["label"] = df["label"].astype(int)

df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"] != ""].copy()

df = df.reset_index(drop=True)

display(df)
print(df.dtypes)
print(df["label"].value_counts())

,text,label
0,"CAIXA, arquivo para documentos, resinada em pa...",0
1,"SACO, plastico para lixo, classe I, capacidade...",0
2,"Aquisição de medicamentos (DICLOFENACO, VITAMI...",0
3,"Aquisição de medicamentos (AMIODARONA, GLICERI...",0
4,CONTRATAÇÃO SERVIÇO DE CONFECÇÃO DE CRACHÁS PE...,1
...,...,...
604,Aquisição e instalação de bobinas térmicas par...,1
605,Prestação de serviços de drones profissionais ...,0
606,Fornecimento parcelado de insumos laboratoriai...,0
607,Locação de serviços de telefonia fixa e móvel ...,0


text       str
label    int64
dtype: object
label
0    305
1    304
Name: count, dtype: int64


# Split treino / validação

In [3]:
dataset = Dataset.from_pandas(df)

dataset = dataset.train_test_split(
    test_size=0.2,
    seed=42
)

# =========================
# 2. Tokenizer
# =========================

In [4]:

MODEL_NAME = "neuralmind/bert-base-portuguese-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

dataset = dataset.map(tokenize, batched=True)
dataset = dataset.remove_columns(["text"])




Map:   0%|          | 0/487 [00:00<?, ? examples/s]

Map:   0%|          | 0/122 [00:00<?, ? examples/s]

# =========================
# 3. Modelo
# =========================

In [5]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. C

# =========================
# 4. Métricas
# =========================

In [6]:


# opcional: só se você quiser logs do TensorBoard
os.environ["TENSORBOARD_LOGGING_DIR"] = "./logs"


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="binary", zero_division=0
    )
    acc = accuracy_score(labels, predictions)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall,
    }

# =========================
# 5. Treinamento
# =========================
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
)
# obter os splits e garantir que são HF Datasets (ajuda o type checker)
# dataset pode ser um DatasetDict; force cast para o type checker
dataset_dict: DatasetDict = cast(DatasetDict, dataset)
train_dataset: HfDataset = cast(HfDataset, dataset_dict['train'])
test_dataset: HfDataset = cast(HfDataset, dataset_dict['test'])
# configurar para retornar tensores torch
train_dataset = train_dataset.with_format("torch")
test_dataset = test_dataset.with_format("torch")

# data collator para padding dinâmico (mais compatível que processing_class)
data_collator = DataCollatorWithPadding(tokenizer)

trainer: HfTrainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

# =========================
# 6. Salvar modelo
# =========================
trainer.save_model("modelo_classificador")
tokenizer.save_pretrained("modelo_classificador")

# =========================
# 7. Inferência
# =========================
def predict(text):
    # garantir que o objeto trainer/modelo exista em tempo de execução
    global trainer
    if trainer is None or getattr(trainer, 'model', None) is None:
        raise RuntimeError('Trainer não inicializado — execute a célula de treino primeiro.')

    # Para o type-checker, faça um cast explícito e use uma variável local 'model'
    from typing import cast
    import torch as _torch
    # obter o atributo model com getattr e garantir que não é None
    model_obj = getattr(trainer, 'model', None)
    assert model_obj is not None, 'trainer.model unexpectedly None'
    model = cast(_torch.nn.Module, model_obj)
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    # determinar device a partir do modelo (type-checker agora reconhece model como nn.Module)
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with _torch.no_grad():
        outputs = model(**inputs)

    # checar se logits são finitos antes do softmax
    if not _torch.isfinite(outputs.logits).all():
        raise FloatingPointError('Logits contêm valores não-finitos (NaN/Inf)')

    probs = _torch.softmax(outputs.logits, dim=1)
    pred = _torch.argmax(probs, dim=1).item()

    return {
        "label": pred,
        "confidence": float(probs.max().item())
    }

/Users/jose-cleiton/dev/script_pncp/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.172307,0.901639,0.911765,0.837838,1.000000
2,No log,0.096087,0.959016,0.957983,1.000000,0.919355
3,No log,0.020480,0.983607,0.983871,0.983871,0.983871


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/jose-cleiton/dev/script_pncp/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/jose-cleiton/dev/script_pncp/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [14]:
value = """
Registro de Preços para contratação de pessoa jurídica para fornecimento de equipamento de ponto eletrônico com leitor biométrico, para marcação de jornada de trabalho por empregado, homol... """

In [15]:
print(predict(value))

{'label': 1, 'confidence': 0.9925111532211304}


In [9]:
from typing import Any, Dict




def avaliar_resultado_modelo(resultado: Dict[str, Any], limite_confianca: float = 0.8) -> Dict[str, Any]:
    """
    Avalia o retorno do modelo com base em:
    - label 1: atende
    - label 0: não atende
    - confidence abaixo do limite: exige validação humana
    """

    if not isinstance(resultado, dict):
        raise TypeError("O resultado do modelo deve ser um dicionário.")

    if "label" not in resultado or "confidence" not in resultado:
        raise ValueError("O resultado do modelo deve conter as chaves 'label' e 'confidence'.")

    label = resultado["label"]
    confidence = float(resultado["confidence"])

    classificacao = "atende" if label == 1 else "nao_atende"

    if confidence < limite_confianca:
        return {
            "status": "validacao_humana",
            "motivo": f"Confiança abaixo do limite mínimo de {limite_confianca:.2f}",
            "classificacao": classificacao,
            "label_modelo": label,
            "confidence": confidence,
        }

    return {
        "status": "aprovado_pelo_modelo",
        "classificacao": classificacao,
        "label_modelo": label,
        "confidence": confidence,
    }


if __name__ == "__main__":
    result = predict(value)
    resposta = avaliar_resultado_modelo(result)
    print(resposta)

{'status': 'aprovado_pelo_modelo', 'classificacao': 'nao_atende', 'label_modelo': 0, 'confidence': 0.9890776872634888}
